## Things I have done in this file:

**Data Schema Standardization:** Corrected column naming conventions and cast data fields to their appropriate statistical types.

**Missing Value Management:** Removed rows with critical missing entries where data recovery wasn't viable, and applied logical imputation to fill missing categorical data based on contextual data patterns.

**Feature Encoding:** Converted text-based categorical variables (e.g., "Yes/No") into proper boolean types for optimal memory usage and model compatibility.

**Anomaly Detection:** Audited columns individually to identify, isolate, and address statistical outliers.

**Deduplication:** Screened the dataset for redundant or duplicated records to ensure data integrity.

In [41]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"data.csv")

In [42]:
df

,Area_SqFt,Rooms,Build_Year,Location,Street_Type,Furnishing,Property_Type,Has_Pool,Price
0,2473.192784,4.0,1992,Jaipur,Residential Lane,Furnished,Apartment,No,568486.0
1,2353.472711,4.0,2006,Indore,Corner Plot,Unfurnished,Apartment,Yes,577214.0
2,2212.222005,3.0,2012,Jaipur,Highway Facing,Semi-Furnished,Duplex,No,581300.0
3,2823.886596,6.0,1993,Lucknow,Main Road,Unfurnished,Villa,Yes,794614.0
4,1869.648721,5.0,2012,Jaipur,Corner Plot,Semi-Furnished,Apartment,No,493086.0
...,...,...,...,...,...,...,...,...,...
1119,2461.048417,7.0,1988,Noida,Corner Plot,Unfurnished,Duplex,No,699925.0
1120,3558.174078,7.0,1989,Kanpur,Highway Facing,Unfurnished,Apartment,No,736329.0
1121,2179.880978,4.0,2011,Prayagraj,Residential Lane,Semi-Furnished,Duplex,No,533498.0
1122,NaN,6.0,2022,Delhi,Main Road,Unfurnished,Independent House,No,533436.0


In [43]:
df.columns = df.columns.str.lower()

In [44]:
df.columns

Index(['area_sqft', 'rooms', 'build_year', 'location', 'street_type',
       'furnishing', 'property_type', 'has_pool', 'price'],
      dtype='object')

In [45]:
df.isnull().sum()

area_sqft        33
rooms            33
build_year        0
location          0
street_type       0
furnishing       33
property_type     0
has_pool          0
price             0
dtype: int64

In [46]:
df = df.dropna(subset=['area_sqft', 'furnishing'])

In [47]:
df.isnull().sum()

area_sqft         0
rooms            32
build_year        0
location          0
street_type       0
furnishing        0
property_type     0
has_pool          0
price             0
dtype: int64

In [56]:
clean_data = df.dropna(subset=['area_sqft', 'rooms'])
sqft_per_room_ratio = (clean_data['area_sqft'] / clean_data['rooms']).median()

estimated_rooms = (df['area_sqft'] / sqft_per_room_ratio).round()

df['rooms'] = df['rooms'].fillna(estimated_rooms)

df['rooms'] = df['rooms'].astype('Int64')

In [49]:
type_mapping = {
    'location': 'category',
    'street_type': 'category',
    'furnishing': 'category',
    'property_type': 'category',
}
df = df.astype(type_mapping)

df['area_sqft'] = pd.to_numeric(df['area_sqft'], errors='coerce')
df['rooms'] = pd.to_numeric(df['rooms'], errors='coerce')
df['price'] = pd.to_numeric(df['price'], errors='coerce')

df['build_year'] = pd.to_numeric(df['build_year'], errors='coerce').astype(int)

In [50]:
df['has_pool'] = df['has_pool'].map({'Yes': 1, 'No': 0})

df['has_pool'] = df['has_pool'].astype('Int64')

In [51]:
df.dtypes

area_sqft         float64
rooms             float64
build_year          int64
location         category
street_type      category
furnishing       category
property_type    category
has_pool            Int64
price             float64
dtype: object

In [52]:
text_columns = ['location', 'street_type', 'furnishing', 'property_type']

for col in text_columns:
    df[col] = df[col].str.lower()
    
df[col] = df[col].astype('category')

In [53]:
df.duplicated().sum()

np.int64(0)

In [59]:
df['area_sqft'].min()

700.0

In [64]:
df['area_sqft'].nlargest(10)

1073    10267.124330
428      9022.275608
840      8858.260499
1056     7808.573894
659      6680.510005
286      6170.583006
52       5648.321783
211      5505.772234
7        4316.818615
339      4122.732784
Name: area_sqft, dtype: float64

In [65]:
df['rooms'].max()

np.int64(7)

In [66]:
df['rooms'].min()

np.int64(2)

In [68]:
df['price'].nlargest(10)

1073    2071401.840
428     1396460.448
659     1366762.171
52      1355674.054
1056    1353997.690
211     1284263.922
840     1273974.917
286     1265986.684
134     1031661.000
7        988286.819
Name: price, dtype: float64

In [69]:
df['price'].min()

248640.0

In [70]:
df['build_year'].min()

1985

In [71]:
df['build_year'].max()

2024

In [72]:
df.describe()

,area_sqft,rooms,build_year,has_pool,price
count,1059.000000,1059.0,1059.000000,1059.0,1.059000e+03
mean,2236.865059,4.562795,2005.118036,0.273843,6.095272e+05
std,736.886722,1.658618,11.758046,0.44614,1.443205e+05
min,700.000000,2.0,1985.000000,0.0,2.486400e+05
25%,1833.266697,3.0,1995.000000,0.0,5.177645e+05
50%,2200.245022,5.0,2006.000000,0.0,6.032640e+05
75%,2575.760949,6.0,2015.000000,1.0,6.896370e+05
max,10267.124330,7.0,2024.000000,1.0,2.071402e+06


In [73]:
df.to_csv("clean_data.csv",index=False)